# Particle Cloud Evolution

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Many particles spreading under Langevin dynamics (Fokker–Planck preview).

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/particle-cloud/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(42)

# The demo's defaults: 300 particles near (0, -4), Langevin toward a two-bump target
n, steps, eta = 300, 1000, 0.05
m = torch.tensor([[-2.0, -1.0], [2.0, 1.5]], dtype=torch.float64)
var = 0.8


def score(x):
    w = torch.softmax(-((x[:, None, :] - m) ** 2).sum(-1) / (2 * var), dim=1)
    return (w[:, :, None] * (-(x[:, None, :] - m) / var)).sum(1)


x = torch.stack([0.3 * torch.randn(n), -4 + 0.3 * torch.randn(n)], 1).double()

# ── Watch the cloud: the empirical distribution flows toward the target ─────
print("  step   mean            fraction near mode 1 / mode 2")
for k in range(steps + 1):
    if k in (0, 30, 100, 300, steps):
        near = ((x[:, None, :] - m) ** 2).sum(-1).argmin(1)
        print(f"{k:6d}   {str([round(v, 2) for v in x.mean(0).tolist()]):14s}  "
              f"{(near == 0).double().mean().item():.2f} / {(near == 1).double().mean().item():.2f}")
    x = x + eta * score(x) + (2 * eta) ** 0.5 * torch.randn_like(x)

# ── Compare with exact samples from the target ──────────────────────────────
pick = torch.randint(0, 2, (n,))
exact = m[pick] + var ** 0.5 * torch.randn(n, 2, dtype=torch.float64)
print("\nfinal cloud mean", [round(v, 3) for v in x.mean(0).tolist()], " exact-sample mean", [round(v, 3) for v in exact.mean(0).tolist()])
print("final cloud std ", [round(v, 3) for v in x.std(0).tolist()], " exact-sample std ", [round(v, 3) for v in exact.std(0).tolist()])

  step   mean            fraction near mode 1 / mode 2
     0   [0.01, -3.97]   1.00 / 0.00
    30   [-1.73, -1.36]  0.96 / 0.04
   100   [-1.61, -0.83]  0.91 / 0.09
   300   [-1.01, -0.35]  0.73 / 0.27
  1000   [-0.31, 0.09]   0.57 / 0.43

final cloud mean [-0.289, 0.103]  exact-sample mean [0.189, 0.305]
final cloud std  [2.289, 1.482]  exact-sample std  [2.211, 1.588]
